# Neighborhood Renovation Investment Targeting

**Original Question:** Where would buying a property and renovating give the best investment opportunity?

_Exported from PlotStudio_

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


# PlotStudio stub — no-op outside the app
def add_to_workspace(df):
    pass


# Load source data
df_housing_invest = pd.read_csv("data/df_housing_invest.csv")
df_neighborhood_reno_counts = pd.read_csv("data/df_neighborhood_reno_counts.csv")
df_neighborhood_premiums = pd.read_csv("data/df_neighborhood_premiums.csv")
df_neighborhood_premium_volatility = pd.read_csv("data/df_neighborhood_premium_volatility.csv")
df_neighborhood_invest_scores = pd.read_csv("data/df_neighborhood_invest_scores.csv")
df_neighborhood_invest_tiers = pd.read_csv("data/df_neighborhood_invest_tiers.csv")
df_neighborhood_premium_summary = pd.read_csv("data/df_neighborhood_premium_summary.csv")

## Task 1: Confirm the structure and sufficiency of existing neighborhood-level renovation and investment datasets, and establish how renovation premiums and baseline prices vary across neighborhoods.

**Acceptance Criteria:** We have a clear understanding of how renovation status and premiums are defined, verified sample sizes per neighborhood, and an initial view of the trade-off between baseline affordability and renovation premium across neighborhoods, supported by at least one concise visualization.

### 1.1: Verify the availability, structure, and key columns of neighborhood-level investment dataframes and restate the analysis sample and renovation group definitions.
_Output: print_

In [ ]:
import pandas as pd
import numpy as np

# Structural inspection
for df_name, df_obj in [
    ("df_housing_invest", df_housing_invest),
    ("df_neighborhood_reno_counts", df_neighborhood_reno_counts),
    ("df_neighborhood_premiums", df_neighborhood_premiums),
    ("df_neighborhood_premium_volatility", df_neighborhood_premium_volatility),
    ("df_neighborhood_invest_scores", df_neighborhood_invest_scores),
    ("df_neighborhood_invest_tiers", df_neighborhood_invest_tiers),
]:
    print("DataFrame:", df_name)
    print("Shape:", df_obj.shape)
    if df_name in ["df_housing_invest", "df_neighborhood_premiums"]:
        print("Columns:", list(df_obj.columns))
    print("Head(5):")
    print(df_obj.head(5))
    print("-" * 80)

# Renovation group inspection
recent_counts = (
    df_housing_invest["is_recent_renovation"].value_counts(dropna=False).sort_index()
)
non_recent_counts = (
    df_housing_invest["is_non_recent_renovation"]
    .value_counts(dropna=False)
    .sort_index()
)
print("is_recent_renovation value counts:")
print(recent_counts)
print("is_non_recent_renovation value counts:")
print(non_recent_counts)

both_flagged = (
    (df_housing_invest["is_recent_renovation"] == 1)
    & (df_housing_invest["is_non_recent_renovation"] == 1)
).sum()
neither_flagged = (
    (df_housing_invest["is_recent_renovation"] == 0)
    & (df_housing_invest["is_non_recent_renovation"] == 0)
).sum()
exactly_one_flagged = (
    (
        (df_housing_invest["is_recent_renovation"] == 1)
        ^ (df_housing_invest["is_non_recent_renovation"] == 1)
    )
).sum()
print("Rows in exactly one renovation group:", int(exactly_one_flagged))
print("Rows in both renovation groups:", int(both_flagged))
print("Rows in neither renovation group:", int(neither_flagged))
if both_flagged == 0 and neither_flagged == 0:
    print("Check: each row falls into exactly one of the two renovation groups.")
else:
    print("Check: a third group or overlap exists.")

# Distinct neighborhood checks
n_unique_reno = df_neighborhood_reno_counts["Neighborhood"].nunique(dropna=False)
n_unique_prem = df_neighborhood_premiums["Neighborhood"].nunique(dropna=False)
print(
    "Distinct Neighborhood values in df_neighborhood_reno_counts:", int(n_unique_reno)
)
print("Distinct Neighborhood values in df_neighborhood_premiums:", int(n_unique_prem))
if n_unique_reno != n_unique_prem:
    print(
        "Mismatch noted: the neighborhood counts and premiums tables cover different neighborhood sets."
    )
else:
    print("No mismatch in distinct Neighborhood counts.")

# Joined sample to verify aggregation logic
if "median_sale_price_per_sqft" in df_housing_invest.columns:
    df_housing_invest_recent = df_housing_invest.loc[
        df_housing_invest["is_recent_renovation"] == 1,
        ["Neighborhood", "sale_price_per_sqft"],
    ].copy()
    df_housing_invest_non_recent = df_housing_invest.loc[
        df_housing_invest["is_non_recent_renovation"] == 1,
        ["Neighborhood", "sale_price_per_sqft"],
    ].copy()

    df_recent_summary = (
        df_housing_invest_recent.groupby("Neighborhood")
        .agg(
            recent_sales=("sale_price_per_sqft", "size"),
            median_sale_price_per_sqft_recent=("sale_price_per_sqft", "median"),
        )
        .reset_index()
    )

    df_non_recent_summary = (
        df_housing_invest_non_recent.groupby("Neighborhood")
        .agg(
            not_recent_sales=("sale_price_per_sqft", "size"),
            median_sale_price_per_sqft_not_recent=("sale_price_per_sqft", "median"),
        )
        .reset_index()
    )

    df_joined_sample = pd.merge(
        df_recent_summary, df_non_recent_summary, on="Neighborhood", how="outer"
    ).head(10)

    print("Joined sample for aggregation verification:")
    print(df_joined_sample)
else:
    df_joined_sample = pd.merge(
        df_neighborhood_reno_counts[
            ["Neighborhood", "recent_sales", "not_recent_sales"]
        ],
        df_neighborhood_premiums[
            [
                "Neighborhood",
                "median_sale_price_per_sqft_recent",
                "median_sale_price_per_sqft_not_recent",
            ]
        ],
        on="Neighborhood",
        how="outer",
    ).head(10)
    print("Joined sample for aggregation verification:")
    print(df_joined_sample)

print("Summary:")
print("Unit of analysis is one property sale in df_housing_invest.")
print("Population is the full set of sales represented in df_housing_invest.")
print(
    "Renovation groups are defined by is_recent_renovation and is_non_recent_renovation."
)
print("Neighborhood aggregations are based on the full df_housing_invest population.")


### 1.2: Quantify, for each neighborhood, the renovation premium and baseline (non-recent) price levels to prepare for ROI-style comparisons.
_Output: print_

In [ ]:
import pandas as pd
import numpy as np

# Build a neighborhood premium summary from the existing premium table
if "has_sufficient_recent_and_not_recent" in df_neighborhood_reno_counts.columns:
    df_neighborhood_sufficiency = df_neighborhood_reno_counts[
        ["Neighborhood", "total_sales", "has_sufficient_recent_and_not_recent"]
    ].copy()
    df_neighborhood_sufficiency["sample_sufficient"] = df_neighborhood_sufficiency[
        "has_sufficient_recent_and_not_recent"
    ].astype(int)
else:
    df_neighborhood_sufficiency = df_neighborhood_reno_counts[
        ["Neighborhood", "total_sales", "recent_sales", "not_recent_sales"]
    ].copy()
    df_neighborhood_sufficiency["sample_sufficient"] = (
        (df_neighborhood_sufficiency["recent_sales"] >= 10)
        & (df_neighborhood_sufficiency["not_recent_sales"] >= 10)
    ).astype(int)

# Keep only the requested neighborhood-level fields and rename them clearly
df_neighborhood_premium_summary = df_neighborhood_premiums[
    [
        "Neighborhood",
        "median_sale_price_per_sqft_not_recent",
        "median_sale_price_per_sqft_recent",
        "median_sale_price_per_sqft_abs_premium",
        "median_sale_price_per_sqft_pct_premium",
        "sales_count_not_recent",
        "sales_count_recent",
    ]
].copy()

df_neighborhood_premium_summary = pd.merge(
    df_neighborhood_premium_summary,
    df_neighborhood_sufficiency,
    on="Neighborhood",
    how="left",
).reset_index(drop=True)

# Standardize names and make the summary easy to read
df_neighborhood_premium_summary = df_neighborhood_premium_summary.rename(
    columns={
        "median_sale_price_per_sqft_not_recent": "median_non_recent_sale_price_per_sqft",
        "median_sale_price_per_sqft_recent": "median_recent_sale_price_per_sqft",
        "median_sale_price_per_sqft_abs_premium": "absolute_premium_per_sqft",
        "median_sale_price_per_sqft_pct_premium": "percentage_premium_per_sqft",
        "sales_count_not_recent": "count_non_recent_sales",
        "sales_count_recent": "count_recent_sales",
    }
)

# Ensure sample sufficiency is always usable
if "sample_sufficient" not in df_neighborhood_premium_summary.columns:
    df_neighborhood_premium_summary["sample_sufficient"] = (
        (df_neighborhood_premium_summary["count_recent_sales"] >= 10)
        & (df_neighborhood_premium_summary["count_non_recent_sales"] >= 10)
    ).astype(int)
else:
    df_neighborhood_premium_summary["sample_sufficient"] = (
        df_neighborhood_premium_summary["sample_sufficient"].fillna(0).astype(int)
    )

# Sort for the main report
if "percentage_premium_per_sqft" in df_neighborhood_premium_summary.columns:
    df_neighborhood_premium_summary = df_neighborhood_premium_summary.sort_values(
        "percentage_premium_per_sqft", ascending=False
    ).reset_index(drop=True)

# Print the full summary
print("Full neighborhood premium summary sorted by percentage premium descending:")
print(df_neighborhood_premium_summary)

# Print top 5 and bottom 5 neighborhoods
print("Top 5 neighborhoods by percentage premium per sqft:")
print(
    df_neighborhood_premium_summary[
        [
            "Neighborhood",
            "percentage_premium_per_sqft",
            "count_non_recent_sales",
            "count_recent_sales",
            "total_sales",
            "sample_sufficient",
        ]
    ].head(5)
)

print("Bottom 5 neighborhoods by percentage premium per sqft:")
print(
    df_neighborhood_premium_summary[
        [
            "Neighborhood",
            "percentage_premium_per_sqft",
            "count_non_recent_sales",
            "count_recent_sales",
            "total_sales",
            "sample_sufficient",
        ]
    ].tail(5)
)

# Short notes on opportunity quality
if "total_sales" in df_neighborhood_premium_summary.columns:
    df_moderate_sample_high_premium = df_neighborhood_premium_summary[
        (df_neighborhood_premium_summary["sample_sufficient"] == 1)
        & (
            df_neighborhood_premium_summary["percentage_premium_per_sqft"]
            >= df_neighborhood_premium_summary["percentage_premium_per_sqft"].median()
        )
    ][
        [
            "Neighborhood",
            "percentage_premium_per_sqft",
            "count_non_recent_sales",
            "count_recent_sales",
            "total_sales",
        ]
    ].copy()
else:
    df_moderate_sample_high_premium = df_neighborhood_premium_summary[
        (df_neighborhood_premium_summary["sample_sufficient"] == 1)
        & (
            df_neighborhood_premium_summary["percentage_premium_per_sqft"]
            >= df_neighborhood_premium_summary["percentage_premium_per_sqft"].median()
        )
    ][
        [
            "Neighborhood",
            "percentage_premium_per_sqft",
            "count_non_recent_sales",
            "count_recent_sales",
        ]
    ].copy()

print("Neighborhoods with relatively high premiums and adequate sample sizes:")
if len(df_moderate_sample_high_premium) == 0:
    print("No neighborhoods met the combined filter in this summary.")
else:
    print(
        df_moderate_sample_high_premium.sort_values(
            "percentage_premium_per_sqft", ascending=False
        ).head(10)
    )

print("Neighborhoods that look thinly sampled and need caution:")
df_thin_sample_caution = df_neighborhood_premium_summary[
    (df_neighborhood_premium_summary["sample_sufficient"] == 0)
    | (df_neighborhood_premium_summary["count_non_recent_sales"] < 10)
    | (df_neighborhood_premium_summary["count_recent_sales"] < 10)
][
    [
        "Neighborhood",
        "percentage_premium_per_sqft",
        "count_non_recent_sales",
        "count_recent_sales",
        "total_sales",
        "sample_sufficient",
    ]
].copy()
if len(df_thin_sample_caution) == 0:
    print("No thinly sampled neighborhoods were flagged by the sufficiency rule.")
else:
    print(
        df_thin_sample_caution.sort_values(
            "percentage_premium_per_sqft", ascending=False
        ).head(10)
    )

# Save to workspace
add_to_workspace(df_neighborhood_premium_summary)


### 1.3: Create a visual that shows how each neighborhood trades off baseline affordability against renovation premium, highlighting which areas look most attractive at a glance.
_Output: figure_

In [ ]:
import pandas as pd
import plotly.express as px

# Merge the existing neighborhood premium summary with opportunity tiers
if (
    "df_neighborhood_premium_summary" in globals()
    and "df_neighborhood_invest_tiers" in globals()
):
    df_neighborhood_invest_view = pd.merge(
        df_neighborhood_premium_summary,
        df_neighborhood_invest_tiers[["Neighborhood", "opportunity_tier"]],
        on="Neighborhood",
        how="left",
    ).reset_index(drop=True)
else:
    raise NameError("Required DataFrames are not available in the namespace.")

# Create the scatter plot
fig = px.scatter(
    df_neighborhood_invest_view,
    x="median_non_recent_sale_price_per_sqft",
    y="percentage_premium_per_sqft",
    color="opportunity_tier",
    size="total_sales",
    hover_name="Neighborhood",
    labels={
        "median_non_recent_sale_price_per_sqft": "Median Non-Recent Sale Price per Sqft (USD)",
        "percentage_premium_per_sqft": "Recent Renovation Premium per Sqft (%)",
        "opportunity_tier": "Opportunity Tier",
        "total_sales": "Total Sales",
    },
    title="Upper-Left Neighborhoods Look Most Attractive: Lower Entry Prices with Higher Renovation Premiums",
)

fig.update_traces(marker_line_color="black", marker_line_width=1)
fig.update_yaxes(tickformat=".1f")
fig.update_xaxes(tickformat=",.0f")
fig.show()


## Task 2: Transform neighborhood-level premiums, baseline prices, and volatility into an ROI- and risk-focused view and explore why some neighborhoods are more attractive opportunities than others.

**Acceptance Criteria:** We obtain a reusable neighborhood ROI dataframe combining premiums, baseline prices, composite scores, and volatility, plus at least one visualization of premium vs volatility and printed interpretation of why certain neighborhoods are classified as higher or lower opportunity.

### 2.1: Create a consolidated neighborhood ROI table that combines premium metrics, baseline prices, volatility, and composite opportunity scores.
_Output: print_

In [ ]:
import pandas as pd

# Build the ROI-focused neighborhood view by joining existing prepared tables

df_neighborhood_roi_view = pd.merge(
    df_neighborhood_premium_summary,
    df_neighborhood_premium_volatility,
    on="Neighborhood",
    how="left",
).reset_index(drop=True)

df_neighborhood_roi_view = pd.merge(
    df_neighborhood_roi_view,
    df_neighborhood_invest_scores[
        [
            "Neighborhood",
            "premium_rank_score",
            "volatility_rank_score",
            "entry_price_rank_score",
            "composite_score",
        ]
    ],
    on="Neighborhood",
    how="left",
).reset_index(drop=True)

df_neighborhood_roi_view = pd.merge(
    df_neighborhood_roi_view,
    df_neighborhood_invest_tiers[["Neighborhood", "opportunity_tier"]],
    on="Neighborhood",
    how="left",
).reset_index(drop=True)

# Make the ROI proxy explicit and readable
# Baseline entry price is the median non-recent price per sqft.
# A simple ROI proxy is the percentage premium per sqft, which reflects uplift relative to entry.
df_neighborhood_roi_view["roi_proxy_pct"] = pd.to_numeric(
    df_neighborhood_roi_view["percentage_premium_per_sqft"], errors="coerce"
)
df_neighborhood_roi_view["roi_proxy_label"] = (
    "Renovation premium vs. baseline entry price"
)

# Ensure consistent ordering for output
sort_cols = ["opportunity_tier", "composite_score", "roi_proxy_pct"]
df_neighborhood_roi_view = df_neighborhood_roi_view.sort_values(
    sort_cols, ascending=[True, False, False]
).reset_index(drop=True)

# Print the resulting view with clear column names
print("Neighborhood ROI view sorted by opportunity tier and composite score:")
print(
    df_neighborhood_roi_view[
        [
            "Neighborhood",
            "median_non_recent_sale_price_per_sqft",
            "median_recent_sale_price_per_sqft",
            "absolute_premium_per_sqft",
            "percentage_premium_per_sqft",
            "roi_proxy_pct",
            "mean_yearly_premium",
            "std_yearly_premium",
            "range_yearly_premium",
            "valid_years",
            "count_non_recent_sales",
            "count_recent_sales",
            "total_sales",
            "premium_rank_score",
            "volatility_rank_score",
            "entry_price_rank_score",
            "composite_score",
            "opportunity_tier",
        ]
    ]
)

# Flag neighborhoods with strong ROI but elevated volatility
if "std_yearly_premium" in df_neighborhood_roi_view.columns:
    df_high_roi_high_volatility = df_neighborhood_roi_view[
        (
            df_neighborhood_roi_view["roi_proxy_pct"]
            >= df_neighborhood_roi_view["roi_proxy_pct"].quantile(0.75)
        )
        & (
            df_neighborhood_roi_view["std_yearly_premium"]
            >= df_neighborhood_roi_view["std_yearly_premium"].quantile(0.75)
        )
    ][
        [
            "Neighborhood",
            "roi_proxy_pct",
            "std_yearly_premium",
            "mean_yearly_premium",
            "composite_score",
            "opportunity_tier",
        ]
    ].copy()
else:
    df_high_roi_high_volatility = df_neighborhood_roi_view.iloc[0:0][
        [
            "Neighborhood",
            "roi_proxy_pct",
            "std_yearly_premium",
            "mean_yearly_premium",
            "composite_score",
            "opportunity_tier",
        ]
    ].copy()

print("High ROI but high volatility neighborhoods:")
if len(df_high_roi_high_volatility) == 0:
    print("No neighborhoods met the combined high-ROI and high-volatility screen.")
else:
    print(
        df_high_roi_high_volatility.sort_values(
            ["roi_proxy_pct", "std_yearly_premium"], ascending=[False, False]
        )
    )

# Flag neighborhoods where expensive entry prices drag down the score
if "entry_price_rank_score" in df_neighborhood_roi_view.columns:
    df_expensive_entry_drag = df_neighborhood_roi_view[
        (
            df_neighborhood_roi_view["entry_price_rank_score"]
            <= df_neighborhood_roi_view["entry_price_rank_score"].quantile(0.25)
        )
        | (
            (
                df_neighborhood_roi_view["entry_price_rank_score"]
                < df_neighborhood_roi_view["premium_rank_score"]
            )
            & (
                df_neighborhood_roi_view["entry_price_rank_score"]
                < df_neighborhood_roi_view["volatility_rank_score"]
            )
        )
    ][
        [
            "Neighborhood",
            "median_non_recent_sale_price_per_sqft",
            "entry_price_rank_score",
            "premium_rank_score",
            "volatility_rank_score",
            "composite_score",
            "opportunity_tier",
        ]
    ].copy()
else:
    df_expensive_entry_drag = df_neighborhood_roi_view.iloc[0:0][
        [
            "Neighborhood",
            "median_non_recent_sale_price_per_sqft",
            "entry_price_rank_score",
            "premium_rank_score",
            "volatility_rank_score",
            "composite_score",
            "opportunity_tier",
        ]
    ].copy()

print("Neighborhoods whose score is dragged down by expensive entry prices:")
if len(df_expensive_entry_drag) == 0:
    print("No clear entry-price drag cases were flagged by the screen.")
else:
    print(df_expensive_entry_drag.sort_values("entry_price_rank_score", ascending=True))

# Save to workspace
add_to_workspace(df_neighborhood_roi_view)


### 2.2: Visually assess the trade-off between renovation premium and volatility across neighborhoods, highlighting risk-adjusted opportunities.
_Output: figure_

In [ ]:
import plotly.express as px

fig = px.scatter(
    df_neighborhood_roi_view,
    x="roi_proxy_pct",
    y="std_yearly_premium",
    color="opportunity_tier",
    size="total_sales",
    hover_name="Neighborhood",
    labels={
        "roi_proxy_pct": "Renovation ROI Proxy (%)",
        "std_yearly_premium": "Premium Volatility (Std. Dev. of Yearly Premium)",
        "opportunity_tier": "Opportunity Tier",
        "total_sales": "Total Sales",
    },
    title="High ROI with Low Volatility Marks the Strongest Risk-Adjusted Neighborhood Opportunities",
)
fig.update_traces(marker_line_color="black", marker_line_width=1)
fig.update_yaxes(tickformat=",.1f")
fig.update_xaxes(tickformat=",.1f")
fig.show()


### 2.3: Explain why some neighborhoods are classified as strong or weak buy-renovate opportunities by decomposing their premium, entry price, and risk characteristics.
_Output: print_

In [ ]:
import pandas as pd

# Work directly from the existing ROI view

df_roi_compact_source = df_neighborhood_roi_view.copy()

# Ensure numeric columns are numeric for safe sorting/printing
for col in [
    "median_non_recent_sale_price_per_sqft",
    "roi_proxy_pct",
    "std_yearly_premium",
    "total_sales",
    "count_recent_sales",
    "count_non_recent_sales",
    "valid_years",
    "composite_score",
    "entry_price_rank_score",
]:
    if col in df_roi_compact_source.columns:
        df_roi_compact_source[col] = pd.to_numeric(
            df_roi_compact_source[col], errors="coerce"
        )

# Select representatives across tiers
# Top three strongest by composite score
if (
    "opportunity_tier" in df_roi_compact_source.columns
    and "composite_score" in df_roi_compact_source.columns
):
    df_top_strong = (
        df_roi_compact_source[df_roi_compact_source["opportunity_tier"] == "strong"]
        .sort_values("composite_score", ascending=False)
        .head(3)
        .copy()
    )
else:
    df_top_strong = df_roi_compact_source.head(3).copy()

# A few middle-tier neighborhoods (promising)
if "opportunity_tier" in df_roi_compact_source.columns:
    df_middle_tier = (
        df_roi_compact_source[df_roi_compact_source["opportunity_tier"] == "promising"]
        .sort_values(["composite_score", "roi_proxy_pct"], ascending=[False, False])
        .head(3)
        .copy()
    )
else:
    df_middle_tier = df_roi_compact_source.iloc[0:0].copy()

# A few caution-tier neighborhoods
if "opportunity_tier" in df_roi_compact_source.columns:
    df_caution_tier = (
        df_roi_compact_source[df_roi_compact_source["opportunity_tier"] == "caution"]
        .sort_values(["composite_score", "roi_proxy_pct"], ascending=[False, False])
        .head(4)
        .copy()
    )
else:
    df_caution_tier = df_roi_compact_source.iloc[0:0].copy()

# Combine and keep a compact set of fields

df_roi_compact = pd.concat(
    [df_top_strong, df_middle_tier, df_caution_tier],
    ignore_index=True,
).reset_index(drop=True)

# Keep a clean presentation order
presentation_cols = [
    "Neighborhood",
    "opportunity_tier",
    "median_non_recent_sale_price_per_sqft",
    "roi_proxy_pct",
    "std_yearly_premium",
    "total_sales",
    "count_recent_sales",
    "count_non_recent_sales",
    "valid_years",
    "composite_score",
]

df_roi_compact = (
    df_roi_compact[presentation_cols]
    .sort_values(["opportunity_tier", "composite_score"], ascending=[True, False])
    .reset_index(drop=True)
)

print("Representative neighborhoods across opportunity tiers:")
print(df_roi_compact)

# Commentary based on the selected neighborhoods
print("Commentary:")
print(
    "The strongest neighborhoods stand out because they combine solid renovation premium with better balance on sample size, volatility, and entry price."
)
print(
    "They tend to score well on composite_score, meaning they are not just high-premium markets; they also avoid being excessively unstable or overpriced at entry."
)
print(
    "Some high-premium neighborhoods still rank lower because the premium is less consistent year to year, or because the baseline non-recent price per sqft is already expensive, which reduces upside from buying and renovating."
)
print(
    "In practice, that means the best buy-renovate targets are the neighborhoods where premium, stability, and entry price all work together rather than where only the premium is high."
)


## Task 3: Integrate premiums, ROI, affordability, and risk to produce a clear, ranked recommendation of neighborhoods that offer the best buy-and-renovate investment opportunities, directly answering the user’s question.

**Acceptance Criteria:** A ranked table and narrative synthesis naming specific neighborhoods as best (and secondary) buy-renovate opportunities, with supporting metrics and explicit caveats about risk and sample sizes, plus a simple ranking visualization and a final validation check.

### 3.1: Identify and rank the neighborhoods that best balance high renovation ROI, reasonable entry price, and acceptable risk, using the consolidated ROI view and opportunity tiers.
_Output: print_

In [ ]:
import pandas as pd
import numpy as np

# Build a robust shortlist from the existing ROI view
required_cols = [
    "Neighborhood",
    "opportunity_tier",
    "median_non_recent_sale_price_per_sqft",
    "roi_proxy_pct",
    "std_yearly_premium",
    "total_sales",
    "count_recent_sales",
    "count_non_recent_sales",
    "valid_years",
    "composite_score",
]

df_best_reno_neighborhoods = df_neighborhood_roi_view.copy()

for col in [
    "median_non_recent_sale_price_per_sqft",
    "roi_proxy_pct",
    "std_yearly_premium",
    "total_sales",
    "count_recent_sales",
    "count_non_recent_sales",
    "valid_years",
    "composite_score",
]:
    df_best_reno_neighborhoods[col] = pd.to_numeric(
        df_best_reno_neighborhoods[col], errors="coerce"
    )

# Basic robustness filters: sufficient counts and enough valid years for comparison
if "sample_sufficient" in df_best_reno_neighborhoods.columns:
    df_best_reno_neighborhoods = df_best_reno_neighborhoods.loc[
        (df_best_reno_neighborhoods["sample_sufficient"] == 1)
        & (df_best_reno_neighborhoods["count_recent_sales"] >= 10)
        & (df_best_reno_neighborhoods["count_non_recent_sales"] >= 10)
        & (df_best_reno_neighborhoods["valid_years"] >= 4)
    ].copy()
else:
    df_best_reno_neighborhoods = df_best_reno_neighborhoods.loc[
        (df_best_reno_neighborhoods["count_recent_sales"] >= 10)
        & (df_best_reno_neighborhoods["count_non_recent_sales"] >= 10)
        & (df_best_reno_neighborhoods["valid_years"] >= 4)
    ].copy()

# Rank by ordered opportunity tier, then composite score, then ROI proxy
opportunity_order = {"strong": 0, "promising": 1, "caution": 2}
df_best_reno_neighborhoods["opportunity_tier_order"] = df_best_reno_neighborhoods[
    "opportunity_tier"
].map(opportunity_order)
df_best_reno_neighborhoods = df_best_reno_neighborhoods.sort_values(
    ["opportunity_tier_order", "composite_score", "roi_proxy_pct"],
    ascending=[True, False, False],
).reset_index(drop=True)

df_best_reno_neighborhoods = df_best_reno_neighborhoods[
    [
        "Neighborhood",
        "opportunity_tier",
        "median_non_recent_sale_price_per_sqft",
        "roi_proxy_pct",
        "std_yearly_premium",
        "total_sales",
        "count_recent_sales",
        "count_non_recent_sales",
        "valid_years",
        "composite_score",
    ]
].copy()

print("Ranked buy-renovate neighborhood shortlist:")
print(df_best_reno_neighborhoods)

print("Top standouts:")
if len(df_best_reno_neighborhoods) > 0:
    df_top_standouts = df_best_reno_neighborhoods.head(3).copy()
    print(df_top_standouts)
else:
    print("No neighborhoods passed the robustness filters.")

print("Caveats:")
print(
    "This shortlist favors neighborhoods with at least 10 recent and 10 non-recent sales and at least 4 valid years of premium comparison."
)
print(
    "Higher ROI proxy is attractive, but neighborhoods with high volatility or expensive entry prices may still be less reliable than their premium suggests."
)
print(
    "The ordered opportunity tier is used first, so strong neighborhoods are prioritized before promising and caution neighborhoods."
)

add_to_workspace(df_best_reno_neighborhoods)


### 3.2: Provide a quick, visual ranking of the top buy-renovate neighborhoods using a single, intuitive chart.
_Output: figure_

In [ ]:
import plotly.express as px

# Prepare a plotting copy with safe numeric types and sorted order
if "df_best_reno_neighborhoods" not in globals():
    raise NameError("df_best_reno_neighborhoods is not available in the namespace.")

df_best_reno_plot = df_best_reno_neighborhoods.copy()
for col in ["composite_score", "roi_proxy_pct"]:
    df_best_reno_plot[col] = pd.to_numeric(df_best_reno_plot[col], errors="coerce")

df_best_reno_plot = df_best_reno_plot.sort_values(
    "composite_score", ascending=True
).reset_index(drop=True)
df_best_reno_plot["roi_proxy_label"] = df_best_reno_plot["roi_proxy_pct"].map(
    lambda v: f"{v:.1f}%" if pd.notna(v) else ""
)

fig = px.bar(
    df_best_reno_plot,
    x="composite_score",
    y="Neighborhood",
    color="opportunity_tier",
    orientation="h",
    text="roi_proxy_label",
    labels={
        "composite_score": "Composite Investment Score",
        "Neighborhood": "Neighborhood",
        "opportunity_tier": "Opportunity Tier",
    },
    title="Neighborhoods Ranked by Composite Investment Score with Renovation Upside",
)
fig.update_traces(
    marker_line_color="black", marker_line_width=1, textposition="outside"
)
fig.update_xaxes(tickformat=".2f")
fig.show()


### 3.3: Cross-check key results for internal consistency and directly answer the user’s question about where buying and renovating offers the best investment opportunity.
_Output: print_

In [ ]:
import pandas as pd
import numpy as np

# Validate shortlist values using only columns that exist in both DataFrames
common_cols = [
    "Neighborhood",
    "opportunity_tier",
    "median_non_recent_sale_price_per_sqft",
    "roi_proxy_pct",
    "std_yearly_premium",
    "composite_score",
    "total_sales",
    "count_recent_sales",
    "count_non_recent_sales",
    "valid_years",
]

if (
    "df_best_reno_neighborhoods" not in globals()
    or "df_neighborhood_roi_view" not in globals()
):
    raise NameError("Required DataFrames are not available in the namespace.")

# Coerce numeric comparison fields safely
for df_obj in [df_best_reno_neighborhoods, df_neighborhood_roi_view]:
    for col in [
        "median_non_recent_sale_price_per_sqft",
        "roi_proxy_pct",
        "std_yearly_premium",
        "composite_score",
        "total_sales",
        "count_recent_sales",
        "count_non_recent_sales",
        "valid_years",
    ]:
        if col in df_obj.columns:
            df_obj[col] = pd.to_numeric(df_obj[col], errors="coerce")

# Build validation table using only common fields

df_shortlist_check = pd.merge(
    df_best_reno_neighborhoods[
        [c for c in common_cols if c in df_best_reno_neighborhoods.columns]
    ].copy(),
    df_neighborhood_roi_view[
        [c for c in common_cols if c in df_neighborhood_roi_view.columns]
    ].copy(),
    on=["Neighborhood"],
    how="left",
    suffixes=("_shortlist", "_roi"),
).reset_index(drop=True)

# Compare only the shared numeric fields that were actually merged with suffixes
comparison_cols = [
    "median_non_recent_sale_price_per_sqft",
    "roi_proxy_pct",
    "std_yearly_premium",
    "composite_score",
]
validation_tolerance = 1e-9

for col in comparison_cols:
    shortlist_col = f"{col}_shortlist"
    roi_col = f"{col}_roi"
    if (
        shortlist_col in df_shortlist_check.columns
        and roi_col in df_shortlist_check.columns
    ):
        df_shortlist_check[f"{col}_diff"] = (
            df_shortlist_check[shortlist_col] - df_shortlist_check[roi_col]
        )
        df_shortlist_check[f"{col}_match"] = (
            df_shortlist_check[f"{col}_diff"].abs() <= validation_tolerance
        )

match_cols = [c for c in df_shortlist_check.columns if c.endswith("_match")]
all_matches = bool(df_shortlist_check[match_cols].all().all()) if match_cols else False

print("Shortlist validation against df_neighborhood_roi_view:")
print(
    df_shortlist_check[
        [
            c
            for c in [
                "Neighborhood",
                "opportunity_tier_shortlist",
                "opportunity_tier_roi",
                "median_non_recent_sale_price_per_sqft_shortlist",
                "median_non_recent_sale_price_per_sqft_roi",
                "roi_proxy_pct_shortlist",
                "roi_proxy_pct_roi",
                "std_yearly_premium_shortlist",
                "std_yearly_premium_roi",
                "composite_score_shortlist",
                "composite_score_roi",
            ]
            if c in df_shortlist_check.columns
        ]
    ]
)
print("Validation status:", "PASS" if all_matches else "CHECK MISMATCHES")

# Compare shortlist vs full universe medians
shortlist_neighborhoods = df_best_reno_neighborhoods["Neighborhood"].tolist()
df_shortlist_universe = df_neighborhood_roi_view[
    df_neighborhood_roi_view["Neighborhood"].isin(shortlist_neighborhoods)
].copy()

df_summary_compare = pd.DataFrame(
    {
        "group": ["shortlist", "full_universe"],
        "median_roi_proxy_pct": [
            pd.to_numeric(
                df_shortlist_universe["roi_proxy_pct"], errors="coerce"
            ).median(),
            pd.to_numeric(
                df_neighborhood_roi_view["roi_proxy_pct"], errors="coerce"
            ).median(),
        ],
        "median_std_yearly_premium": [
            pd.to_numeric(
                df_shortlist_universe["std_yearly_premium"], errors="coerce"
            ).median(),
            pd.to_numeric(
                df_neighborhood_roi_view["std_yearly_premium"], errors="coerce"
            ).median(),
        ],
        "median_composite_score": [
            pd.to_numeric(
                df_shortlist_universe["composite_score"], errors="coerce"
            ).median(),
            pd.to_numeric(
                df_neighborhood_roi_view["composite_score"], errors="coerce"
            ).median(),
        ],
        "n_neighborhoods": [len(df_shortlist_universe), len(df_neighborhood_roi_view)],
    }
)

print("Shortlist versus full ROI universe medians:")
print(df_summary_compare)

# Flag neighborhoods that look attractive on premium alone but weaken after risk or entry price consideration
if {"premium_rank_score", "volatility_rank_score", "entry_price_rank_score"}.issubset(
    df_neighborhood_roi_view.columns
):
    df_appealing_on_premium_but_weaker = df_neighborhood_roi_view[
        (
            df_neighborhood_roi_view["premium_rank_score"]
            >= df_neighborhood_roi_view["premium_rank_score"].median()
        )
        & (
            (
                df_neighborhood_roi_view["volatility_rank_score"]
                < df_neighborhood_roi_view["volatility_rank_score"].median()
            )
            | (
                df_neighborhood_roi_view["entry_price_rank_score"]
                < df_neighborhood_roi_view["entry_price_rank_score"].median()
            )
        )
    ][
        [
            "Neighborhood",
            "opportunity_tier",
            "roi_proxy_pct",
            "std_yearly_premium",
            "entry_price_rank_score",
            "premium_rank_score",
            "volatility_rank_score",
            "composite_score",
        ]
    ].copy()
else:
    df_appealing_on_premium_but_weaker = df_neighborhood_roi_view.iloc[0:0][
        [
            "Neighborhood",
            "opportunity_tier",
            "roi_proxy_pct",
            "std_yearly_premium",
            "entry_price_rank_score",
            "premium_rank_score",
            "volatility_rank_score",
            "composite_score",
        ]
    ].copy()

print("Appealing on premium alone but weaker after risk or entry price adjustment:")
if len(df_appealing_on_premium_but_weaker) == 0:
    print("None flagged by the screening rule.")
else:
    print(
        df_appealing_on_premium_but_weaker.sort_values(
            ["premium_rank_score", "composite_score"], ascending=[False, True]
        )
    )

# Concise synthesis
print("Synthesis:")
print(
    "Best neighborhoods: NWAmes, NAmes, and Sawyer remain the strongest buy-renovate candidates because they pair solid ROI proxy with favorable composite scores."
)
print(
    "Secondary options: SawyerW, Gilbert, and OldTown are viable but deserve more caution because their risk-adjusted profiles are less compelling than the top three."
)
print(
    "Premium-only traps: Timber and Crawfor stand out on renovation premium, but their volatility and/or entry-price profile weakens the case once risk is considered."
)
print(
    "Bottom line: prioritize the shortlist leaders first, then use the promising tier as backup when local comps and renovation budgets support the assumed upside."
)
